# 04 — Feature Store Setup with Feast

**Objective**: Register the gold-layer engineered features (from NB03) in a [Feast](https://feast.dev/) feature store for reproducible feature serving, point-in-time retrieval, and online/offline consistency.

## What Feast Provides

| Capability | Benefit |
|-----------|--------|
| **Feature catalog** | Central registry of all features with metadata |
| **Point-in-time joins** | Prevents data leakage in historical retrieval |
| **Online serving** | Low-latency feature lookup for real-time inference |
| **Offline store** | Batch retrieval for training and batch scoring |

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings("ignore")

# Project paths
PROJECT_ROOT = Path("..").resolve()
FEATURES_DIR = PROJECT_ROOT / "data" / "features"       # Gold layer
FEAST_DIR = PROJECT_ROOT / "data" / "feast"              # Feast data
FEATURE_STORE_DIR = PROJECT_ROOT / "feature_store"       # Feast repo config

FEAST_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:      {PROJECT_ROOT}")
print(f"Gold layer:        {FEATURES_DIR}")
print(f"Feast data:        {FEAST_DIR}")
print(f"Feature store repo: {FEATURE_STORE_DIR}")
print(f"\nGold layer files:")
for f in sorted(FEATURES_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:>8.1f} MB")

## 1. Setup — Load Gold Layer Features

In [ ]:
# Load gold layer training features produced by NB03
train_features = pd.read_parquet(FEATURES_DIR / "train_features.parquet")

print(f"Train features shape: {train_features.shape[0]:,} rows x {train_features.shape[1]} columns")
print(f"\nTarget distribution:")
print(train_features["TARGET"].value_counts())
print(f"\nFirst 5 columns: {list(train_features.columns[:5])}")
print(f"Last 5 columns:  {list(train_features.columns[-5:])}")
print(f"\nKey column (SK_ID_CURR) unique values: {train_features['SK_ID_CURR'].nunique():,}")

train_features.head(3)

## 2. Prepare Data for Feast

Feast requires:
- An **event_timestamp** column for point-in-time correctness
- A **created** column to track when the feature row was generated

Since this is a static Kaggle dataset (all features are point-in-time at application), we use a fixed timestamp.

In [ ]:
# Add timestamp columns required by Feast
feast_df = train_features.copy()

# Use a fixed date — all features represent application-time snapshots
feast_df["event_timestamp"] = pd.Timestamp("2024-01-01", tz="UTC")
feast_df["created"] = pd.Timestamp("2024-01-01", tz="UTC")

# Ensure SK_ID_CURR is int64 (Feast entity key)
feast_df["SK_ID_CURR"] = feast_df["SK_ID_CURR"].astype("int64")

print(f"Feast-ready DataFrame: {feast_df.shape[0]:,} rows x {feast_df.shape[1]} columns")
print(f"\nAdded columns:")
print(f"  event_timestamp: {feast_df['event_timestamp'].dtype} — {feast_df['event_timestamp'].iloc[0]}")
print(f"  created:         {feast_df['created'].dtype} — {feast_df['created'].iloc[0]}")

# Save to Feast data directory
feast_parquet_path = FEAST_DIR / "train_features.parquet"
feast_df.to_parquet(feast_parquet_path, engine="pyarrow", index=False)

size_mb = feast_parquet_path.stat().st_size / 1e6
print(f"\nSaved Feast-ready features: {feast_parquet_path}")
print(f"  Size: {size_mb:.1f} MB")

## 3. Feature Store Configuration

The feature store is configured via two files in `feature_store/`:

### `feature_store.yaml` — Feast repo config

In [ ]:
# Display the feature store configuration
yaml_path = FEATURE_STORE_DIR / "feature_store.yaml"
print(f"Config file: {yaml_path}")
print(f"{'='*60}")
print(yaml_path.read_text())

### `definitions.py` — Entity & Feature View definitions

In [ ]:
# Display the feature definitions
defs_path = FEATURE_STORE_DIR / "definitions.py"
print(f"Definitions file: {defs_path}")
print(f"{'='*60}")
print(defs_path.read_text())

## 4. Apply Feature Store

Register the entity, data sources, and feature views with the Feast registry.

In [ ]:
import sys
sys.path.insert(0, str(FEATURE_STORE_DIR))

from feast import FeatureStore
from definitions import (
    applicant,
    train_source,
    applicant_features,
    bureau_features,
    credit_history_features,
)

# Initialize the feature store (repo_path points to directory with feature_store.yaml)
fs = FeatureStore(repo_path=str(FEATURE_STORE_DIR))

# Apply all definitions to the registry
fs.apply([
    applicant,
    train_source,
    applicant_features,
    bureau_features,
    credit_history_features,
])

print("Feature store applied successfully!")
print(f"\nRegistry location: {FEAST_DIR / 'registry.db'}")
print(f"\nRegistered entities:")
for entity in fs.list_entities():
    print(f"  - {entity.name} (join_keys={entity.join_keys})")

print(f"\nRegistered feature views:")
for fv in fs.list_feature_views():
    n_features = len(fv.features)
    print(f"  - {fv.name:30s} ({n_features:>2} features)")

## 5. Materialize Features to Online Store

Materialize features into the online store (SQLite) for low-latency serving.

In [ ]:
from datetime import datetime, timezone

# Materialize features into the online store
# Use a time range that covers our event_timestamp (2024-01-01)
start_date = datetime(2023, 12, 31, tzinfo=timezone.utc)
end_date = datetime(2024, 1, 2, tzinfo=timezone.utc)

print(f"Materializing features from {start_date} to {end_date} ...")
fs.materialize(
    start_date=start_date,
    end_date=end_date,
)
print("\nMaterialization complete!")

# Check online store file
online_store_path = FEAST_DIR / "online_store.db"
if online_store_path.exists():
    size_mb = online_store_path.stat().st_size / 1e6
    print(f"Online store: {online_store_path} ({size_mb:.1f} MB)")

## 6. Retrieve Features

Demonstrate both **historical** (offline) and **online** feature retrieval.

### 6.1 Historical (Offline) Retrieval

Point-in-time correct feature retrieval for training or batch scoring.

In [ ]:
# Create entity DataFrame for retrieval — sample 10 applicants
sample_ids = feast_df["SK_ID_CURR"].sample(n=10, random_state=42).tolist()

entity_df = pd.DataFrame({
    "SK_ID_CURR": sample_ids,
    "event_timestamp": pd.Timestamp("2024-01-01", tz="UTC"),
})

print("Entity DataFrame for retrieval:")
print(entity_df.to_string(index=False))

In [ ]:
# Retrieve historical features from offline store
feature_refs = [
    "applicant_features:AMT_INCOME_TOTAL",
    "applicant_features:AMT_CREDIT",
    "applicant_features:AMT_ANNUITY",
    "applicant_features:AGE_YEARS",
    "applicant_features:EXT_SOURCE_1",
    "applicant_features:EXT_SOURCE_2",
    "applicant_features:EXT_SOURCE_3",
    "applicant_features:CREDIT_INCOME_RATIO",
    "bureau_features:BUREAU_LOAN_COUNT",
    "bureau_features:BUREAU_AMT_CREDIT_SUM_SUM",
    "bureau_features:BUREAU_ACTIVE_LOANS_PCT",
    "credit_history_features:PREV_APP_COUNT",
    "credit_history_features:PREV_APPROVED_COUNT",
    "credit_history_features:INSTAL_PAYMENT_DIFF_MEAN",
]

historical_features = fs.get_historical_features(
    entity_df=entity_df,
    features=feature_refs,
).to_df()

print(f"Retrieved historical features: {historical_features.shape[0]} rows x {historical_features.shape[1]} columns")
print(f"\nColumns: {list(historical_features.columns)}")
historical_features

### 6.2 Online Retrieval

Low-latency feature lookup for real-time inference (from materialized online store).

In [ ]:
# Retrieve features from the online store for a single applicant
online_features = fs.get_online_features(
    features=[
        "applicant_features:AMT_INCOME_TOTAL",
        "applicant_features:AMT_CREDIT",
        "applicant_features:AGE_YEARS",
        "applicant_features:EXT_SOURCE_2",
        "bureau_features:BUREAU_LOAN_COUNT",
        "credit_history_features:PREV_APP_COUNT",
    ],
    entity_rows=[{"SK_ID_CURR": sample_ids[0]}],
).to_dict()

print(f"Online features for SK_ID_CURR = {sample_ids[0]}:")
print(f"{'-'*50}")
for key, values in online_features.items():
    val = values[0]
    if isinstance(val, float):
        print(f"  {key:40s} {val:>12.2f}")
    else:
        print(f"  {key:40s} {str(val):>12s}")

### 6.3 Validate Retrieval Consistency

Verify that features retrieved from Feast match the original gold layer data.

In [ ]:
# Compare Feast-retrieved features with original gold layer
check_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AGE_YEARS"]

# Get original values for the sample IDs
original = train_features[train_features["SK_ID_CURR"].isin(sample_ids)].set_index("SK_ID_CURR")
retrieved = historical_features.set_index("SK_ID_CURR")

print("Consistency check (Feast vs Gold Layer):")
print(f"{'Column':40s} {'Match':>8s}")
print(f"{'-'*50}")

all_match = True
for col in check_cols:
    orig_vals = original[col].sort_index().values
    feast_vals = retrieved[col].sort_index().values
    match = np.allclose(orig_vals, feast_vals, equal_nan=True)
    all_match = all_match and match
    print(f"  {col:38s} {'OK' if match else 'MISMATCH':>8s}")

print(f"\n{'All features match!' if all_match else 'ERRORS DETECTED — check feature definitions'}")

## Summary

**What we did:**
1. Loaded gold-layer engineered features from NB03 (`data/features/train_features.parquet`)
2. Prepared data for Feast by adding `event_timestamp` and `created` columns
3. Configured the Feast feature store (`feature_store.yaml` + `definitions.py`)
4. Registered 1 entity (`applicant`) and 3 feature views:
   - `applicant_features` — core application-level features
   - `bureau_features` — credit bureau aggregation features
   - `credit_history_features` — previous app, POS, credit card, installment features
5. Materialized features to the online store (SQLite) for low-latency serving
6. Demonstrated historical and online feature retrieval
7. Validated that retrieved features match the original gold layer data

**Feature store artifacts:**
- Registry: `data/feast/registry.db`
- Online store: `data/feast/online_store.db`
- Feast-ready features: `data/feast/train_features.parquet`

**Next:** NB05 — Baseline Models (LogReg, Random Forest) + MLFlow experiment tracking